In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as  px

In [2]:
# data sets 
delivery = pd.read_csv('C:/Users/Lenovo/Desktop/Python/Pandas/datas/ipl_deliveries.csv')
matches = pd.read_csv('C:/Users/Lenovo/Desktop/Python/Pandas/datas/ipl-matches.csv')
tips = px.data.tips()

In [3]:
ipl = matches.merge(delivery ,left_on='ID' ,right_on='ID')
ipl.columns

Index(['ID', 'City', 'Date', 'Season', 'MatchNumber', 'Team1', 'Team2',
       'Venue', 'TossWinner', 'TossDecision', 'SuperOver', 'WinningTeam',
       'WonBy', 'Margin', 'method', 'Player_of_Match', 'Team1Players',
       'Team2Players', 'Umpire1', 'Umpire2', 'Team', 'innings', 'overs',
       'ballnumber', 'batter', 'bowler', 'non-striker', 'extra_type',
       'batsman_run', 'extras_run', 'total_run', 'non_boundary',
       'isWicketDelivery', 'player_out', 'kind', 'fielders_involved',
       'BattingTeam', 'BowlingTeam'],
      dtype='str')

In [4]:
# SCATTER PLOT B/W Batsmen AVG VS Strike Rate of top 50 batsmen
top_50 =ipl.groupby('batter')['batsman_run'].sum().sort_values(ascending=False).head(50).index.to_list()

In [5]:
temp_df = ipl[ipl['batter'].isin(top_50)]

In [6]:
runs = temp_df.groupby('batter')['batsman_run'].sum()
balls = temp_df[~temp_df['extra_type'].isin(['wides','noballs'])].groupby('batter')['batsman_run'].count()

In [7]:
sr = ((runs / balls)*100).reset_index()
sr

,batter,batsman_run
0,AB de Villiers,152.382353
1,AC Gilchrist,139.514498
2,AD Russell,179.647577
3,AJ Finch,128.659287
4,AM Rahane,120.818505
5,AT Rayudu,127.821843
6,BB McCullum,132.444853
7,CH Gayle,150.015011
8,DA Miller,138.154192
9,DA Warner,141.214594


In [8]:
avg = (runs /  temp_df['player_out'].value_counts()).reset_index()
avg.rename(columns={0:'avg'} ,inplace=True)

In [9]:
top_50_sr_VS_avg = avg.merge(sr ,left_on='batter' ,right_on='batter')

In [10]:
top_50_sr_VS_avg

,batter,avg,batsman_run
0,AB de Villiers,40.795276,152.382353
1,AC Gilchrist,27.586667,139.514498
2,AD Russell,31.369231,179.647577
3,AJ Finch,24.904762,128.659287
4,AM Rahane,31.581395,120.818505
5,AT Rayudu,29.097222,127.821843
6,BB McCullum,27.980583,132.444853
7,CH Gayle,39.658730,150.015011
8,DA Miller,36.641791,138.154192
9,DA Warner,42.323741,141.214594


In [11]:
# SCATTER PLOT 
trace1 = go.Scatter(x=top_50_sr_VS_avg['avg'] ,y=top_50_sr_VS_avg['batsman_run'],
                    mode='markers',text=top_50_sr_VS_avg['batter'],marker={'colorscale':'Viridis' ,'size':8}
                   ,marker_color=np.arange(50))
data=[trace1]
layout = go.Layout(title='Top 50 ipl batsman avg VS strike Rate' ,xaxis={'title':'avg'},yaxis={'title':'strike_rate'})
fig = go.Figure(data,layout)
fig.show()

### IPL Batsman run year over year line plot

In [12]:
ipl['Year']=ipl['Season'].str.split('/').str[0]

In [13]:
# for V Kohli
bat_df = ipl[ipl['batter'] == 'V Kohli']
ser= bat_df.groupby('Year')['batsman_run'].sum()


In [14]:
trace=go.Scatter(x=ser.index , y=ser.values)
layout= go.Layout(title='V Kohli year over year score in IPL(2007 to 2022)' ,xaxis=dict(title='Year'),yaxis=dict(title='Runs'))
data=[trace]
fig = go.Figure(data , layout)
fig.show()

In [15]:
ipl['batter'].unique()

<ArrowStringArray>
[    'YBK Jaiswal',      'JC Buttler',       'SV Samson',      'D Padikkal',
      'SO Hetmyer',        'R Ashwin',         'R Parag',        'TA Boult',
        'OC McCoy',         'WP Saha',
 ...
         'T Taibu',       'RR Sarwan',       'VRV Singh',          'U Kaul',
   'S Chanderpaul', 'D Kalyankrishna',        'MA Khote',        'SB Joshi',
      'DS Lehmann',       'AA Noffke']
Length: 605, dtype: str

In [16]:
bat_df1 = ipl[ipl['batter'] == 'V Kohli']
ser1= bat_df1.groupby('Year')['batsman_run'].sum()
bat_df2 = ipl[ipl['batter'] == 'KL Rahul']
ser2= bat_df2.groupby('Year')['batsman_run'].sum()

In [17]:
trace1=go.Scatter(x=ser1.index , y=ser1.values ,name='V Kohli')
trace2 = go.Scatter(x=ser2.index , y=ser2.values , name='KL Rahul')
layout= go.Layout(title='V Kohli & KL Rahul year over year score in IPL(2007 to 2022)' ,xaxis=dict(title='Year'),yaxis=dict(title='Runs'))
data=[trace1 ,trace2]
fig = go.Figure(data , layout)
fig.show()

In [18]:
def batsmen_plot(*args)->list: # batsmen_traces
    data=list()
    for i in args:
        single = ipl[ipl['batter'] == i]
        single_ser = single.groupby('Year')['batsman_run'].sum()
        trace= go.Scatter(x=single_ser.index ,y=single_ser.values ,name=i)
        data.append(trace)
    return data

In [19]:
layout= go.Layout(title='Year over Year Batsmen Runs in IPL(2007 to 2022)' ,xaxis=dict(title='Year'),yaxis=dict(title='Runs'))
data=batsmen_plot('V Kohli','KL Rahul','MS Dhoni','DA Warner')
fig = go.Figure(data , layout)
fig.show()

In [20]:
top_5 = ipl.groupby('batter')['batsman_run'].sum().sort_values(ascending=False).head(5).index.to_list()
top_5 = ipl[ipl['batter'].isin(top_5)]

In [21]:
top_5 = top_5[top_5['batsman_run'].isin([4,6])]
fours = top_5[top_5['batsman_run'] == 4].groupby('batter')['batsman_run'].sum().reset_index()
sixes= top_5[top_5['batsman_run'] == 6].groupby('batter')['batsman_run'].sum().reset_index()
top_5 = fours.merge(sixes ,left_on='batter' ,right_on='batter')
top_5.rename(columns={"batsman_run_x":"fours_runs" , "batsman_run_y":"sixes_runs"},inplace=True)

#### TOP5 BATSMAN TOTAL SIXES AND 4 score

In [22]:
trace1 = go.Bar(x=top_5['batter'] ,y=top_5['fours_runs'] ,name='fours runs')
trace2 = go.Bar(x=top_5['batter'] ,y=top_5['sixes_runs'] ,name='sixes runs')
data = [trace1 ,trace2]
layout = go.Layout(title='top5 batsmen sixes & fours' ,xaxis={'title' :'Batsman'} 
                   ,yaxis={'title':'Score'} ,barmode='overlay') # one over the other bars
fig=go.Figure(data,layout)
fig.show()

In [23]:
trace1 = go.Bar(x=top_5['batter'] ,y=top_5['fours_runs'] ,name='fours runs')
trace2 = go.Bar(x=top_5['batter'] ,y=top_5['sixes_runs'] ,name='sixes runs')
data = [trace1 ,trace2]
layout = go.Layout(title='top5 batsmen sixes & fours' ,xaxis={'title' :'Batsman'} 
                   ,yaxis={'title':'Score'} ,barmode='stack') # stack bars
fig=go.Figure(data,layout)
fig.show()

In [24]:
trace1 = go.Bar(x=top_5['batter'] ,y=top_5['fours_runs'] ,name='fours runs')
trace2 = go.Bar(x=top_5['batter'] ,y=top_5['sixes_runs'] ,name='sixes runs')
data = [trace1 ,trace2]
layout = go.Layout(title='top5 batsmen sixes & fours' ,xaxis={'title' :'Batsman'} 
                   ,yaxis={'title':'Score'} ) # by default barmode= 'nested'
fig=go.Figure(data,layout)
fig.show()

### Bubble Plot (batsman avg VS strike rate with size representing number of sixes)

In [26]:
top_50_sr_VS_avg_Size = top_50_sr_VS_avg.merge(temp_df[temp_df['batsman_run'] == 6].groupby('batter')['batsman_run'].count().reset_index() ,left_on='batter' ,right_on='batter')

In [27]:
top_50_sr_VS_avg_Size.rename(columns={"batsman_run_x":'strike_rate' , 'batsman_run_y':'sixes'} ,inplace=True)

In [28]:
# Format hover text into a 1D Series
hover_text = (
    "<b>Batter:</b> " + top_50_sr_VS_avg_Size['batter'].astype(str) + 
    "<br><b>Sixes:</b> " + top_50_sr_VS_avg_Size['sixes'].astype(str)
)


In [29]:
# SCATTER PLOT 
trace1 = go.Scatter(x=top_50_sr_VS_avg_Size['avg'] ,y=top_50_sr_VS_avg_Size['strike_rate'],
                    mode='markers',text=hover_text,
                marker=dict(color=np.arange(50),
            colorscale='Viridis', # by default viridis
            size=top_50_sr_VS_avg_Size['sixes']%10+8)
                    )
data=[trace1]
layout = go.Layout(title='Top 50 ipl batsman avg VS strike Rate' ,xaxis={'title':'avg'},yaxis={'title':'strike_rate'})
fig = go.Figure(data,layout)
fig.show()

In [30]:
ipl_score = ipl.groupby(['ID','Year'])['total_run'].sum().reset_index()

In [31]:
trace = go.Box(x=ipl_score['total_run'], name='All Seasons' , marker = {'color':'green'})
layout = go.Layout(title='Ipl match wise score analysis' ,xaxis={'title':'Score'})
data= [trace]
fig = go.Figure(data , layout)
fig.show()

In [32]:
trace1 = go.Box(x=ipl_score[ipl_score['Year'] == '2021']['total_run'], name='Season 2021')
trace2 = go.Box(x=ipl_score[ipl_score['Year'] == '2022']['total_run'], name='Season 2022')
layout = go.Layout(title='Ipl match wise score analysis' ,xaxis={'title':'Score'} )
data= [trace1,trace2]
fig = go.Figure(data , layout)
fig.show()

In [33]:
trace1 = go.Box(x=tips[tips['smoker']== 'No']['day'] ,y=tips[tips['smoker']== 'No']['total_bill'] ,name='Non smokers')
trace2 = go.Box(x=tips[tips['smoker']== 'Yes']['day'] ,y=tips[tips['smoker']== 'Yes']['total_bill'],name='Smokers' )
layout = go.Layout(title='Smokers Vs Non smokers total bill analysis' ,xaxis={'title':'day'} ,yaxis={'title':'bill'} ,boxmode='group')
data= [trace1 ,trace2]
fig = go.Figure(data , layout)
fig.show()

In [34]:
import plotly.express as px

In [35]:
top_50_sr_VS_avg.head(3)

,batter,avg,batsman_run
0,AB de Villiers,40.795276,152.382353
1,AC Gilchrist,27.586667,139.514498
2,AD Russell,31.369231,179.647577


In [36]:
fig = px.histogram(
    x=top_50_sr_VS_avg['avg'],
    marginal="rug",  # Options: 'rug', 'box', or 'violin'
    nbins=30,
    title="Distribution Plot",
    labels={'x': 'Average'}
)
fig.show()

In [37]:
len(np.unique(ipl[['Team1','Team2']]))

18

In [38]:
# OR 
pd.concat([ipl['Team1'] ,ipl['Team2']]).nunique()

18

### Heatmaps in go

In [39]:
temp_df2 = ipl[ipl['MatchNumber'].isin(['Qualifier 1','Eliminator','Qualifier 2','Final'])]
inning_1 = temp_df2[temp_df2['innings'] == 1]
inning_2 = temp_df2[temp_df2['innings'] == 2]
inning_1 = inning_1.groupby(['MatchNumber','overs'])['total_run'].sum()
inning_2 = inning_2.groupby(['MatchNumber','overs'])['total_run'].sum()

In [40]:
temp_df2 = inning_1.reset_index().merge(inning_2.reset_index(),left_on=['MatchNumber','overs'],right_on=['MatchNumber','overs'])
temp_df2.rename(columns={'total_run_x':'1st_innings' , 'total_run_y':'2nd_innings'},inplace=True)

In [41]:
order = ['Qualifier 1','Qualifier 2' ,'Eliminator' ,'Final']
sorting_technique = pd.Categorical(temp_df2['MatchNumber'] ,categories=order ,ordered=True)
temp_df2['MatchNumber'] = pd.Series(sorting_technique)

In [42]:
temp_df2.sort_values(['MatchNumber' ,'overs'] ,inplace=True)

In [43]:
trace1 = go.Heatmap(x=temp_df2['MatchNumber'] ,y=temp_df2['overs'] ,z=temp_df2[['1st_innings' ,'2nd_innings']].max(axis=1))
layout = go.Layout(title='Max score in last matches of IPL' ,xaxis={'title':'Match'} ,yaxis={'title':'overs'})
data= [trace1 ]
fig = go.Figure(data , layout)
fig.show()

### 3D surface plot
#### only in GO not px(express)

In [44]:
x = np.linspace(-10,10,100)
y = np.linspace(-10,10,100)
X,Y = np.meshgrid(x,y)
Z= X**2 + Y**2
trace1 = go.Surface(x=x,y=y ,z=Z   ,colorscale='Ice')
data = [trace1]
layout = go.Layout(title='x^2 + y^2 Graph' , scene= dict(xaxis_title='X', # for 3d plot title live inside scene
                                                         yaxis_title= 'Y' , zaxis_title= 'Z')
                   )
fig = go.Figure(data ,layout)
fig.show()

In [45]:
trace = go.Contour(x=x ,y=y ,z=Z) # x and y 1d in plotly Contour plot unlike 1d or 2d in surface plot 
data = [trace]
layout = go.Layout(title='Contour plot of z= x^2 + y^2')

fig = go.Figure(data ,layout)
fig.show()

In [46]:
from plotly.subplots import make_subplots

In [47]:
fig = make_subplots(rows=1 , cols=2 ,
                    subplot_titles=("First Subplot","Second Subplot"))
fig.add_trace(go.Scatter(x=[1,2,3,4,5,6,] ,y=[1,2,3,4,9,14] , name='scatterplot') ,
            row=1, col= 1)
fig.add_trace(go.Contour(x=x,y=y,z=Z,name='contour plot') ,
              row=1 , col=2)
fig.update_layout(title = "Subplots demo") # ,showlegend=True)
fig.show()

In [48]:
fig = make_subplots(rows=1, cols=2)

fig.add_trace(
    go.Scatter(
        x=[1, 2, 3],
        y=[4, 5, 6],
        mode="markers+text",
        text=["Text A"],
        textposition="bottom center"
    ),
    row=1, col=1
)
fig.show()